# 03b - GNN explainers: GNNExplainer, Integrated Gradients, GraphLIME

Loads the two GNNs trained in 02b and explains the same 100 test nodes
(50 fraud + 50 legit) with three methods each.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
from torch_geometric.data import Data

from xai_blockchain_framework.config import PATHS, set_seed
from xai_blockchain_framework.models.gnn import TemporalGCN, GraphSAGEModel, get_device
from xai_blockchain_framework.xai.gnn_explainers import (
    run_gnnexplainer, run_ig, run_graphlime,
)
set_seed()

DEVICE = get_device()
FIGURES_PATH = str(PATHS.figures_dir)
RESULTS_PATH = str(PATHS.results_dir)
MODELS_PATH = str(PATHS.models_dir)
PROCESSED_PATH = str(PATHS.data_processed)

N_EXPLAIN = 100
SEED = 42
np.random.seed(SEED)
print(f'Device: {DEVICE}, N_EXPLAIN={N_EXPLAIN}')


## Load the graph and the trained GNNs

In [ ]:
splits = np.load(os.path.join(PROCESSED_PATH, 'elliptic_splits.npz'))
X_all_n = splits['X_all_n']
y_all = splits['y_all']
train_mask = splits['train_mask']
val_mask = splits['val_mask']
test_mask = splits['test_mask']
feature_cols = list(np.load(os.path.join(RESULTS_PATH, 'elliptic_feature_names.npy'), allow_pickle=True))
edge_index = torch.load(os.path.join(PROCESSED_PATH, 'elliptic_edge_index.pt'), weights_only=False)

data = Data(
    x=torch.tensor(X_all_n, dtype=torch.float32),
    y=torch.tensor(y_all, dtype=torch.long),
    edge_index=edge_index,
    train_mask=torch.tensor(train_mask),
    val_mask=torch.tensor(val_mask),
    test_mask=torch.tensor(test_mask),
).to(DEVICE)
print(f'Nodes: {data.num_nodes}, Edges: {data.num_edges}, Features: {data.num_features}')

n_feat = len(feature_cols)
tgcn = TemporalGCN(in_c=n_feat).to(DEVICE)
tgcn.load_state_dict(torch.load(os.path.join(MODELS_PATH, 'elliptic_temporal_gcn.pt'), map_location=DEVICE))
tgcn.eval()

sage = GraphSAGEModel(in_c=n_feat).to(DEVICE)
sage.load_state_dict(torch.load(os.path.join(MODELS_PATH, 'elliptic_graphsage.pt'), map_location=DEVICE))
sage.eval()


## Sample 100 test nodes (50 fraud + 50 legit)

In [ ]:
test_indices = torch.where(data.test_mask)[0]
test_labels = data.y[test_indices]
fraud_nodes = test_indices[test_labels == 1]
legit_nodes = test_indices[test_labels == 0]
rng = np.random.RandomState(SEED)
nodes_to_explain = torch.cat([
    fraud_nodes[rng.choice(len(fraud_nodes), min(N_EXPLAIN // 2, len(fraud_nodes)), replace=False)],
    legit_nodes[rng.choice(len(legit_nodes), min(N_EXPLAIN // 2, len(legit_nodes)), replace=False)],
])
np.save(os.path.join(RESULTS_PATH, 'gnn_eval_node_indices.npy'), nodes_to_explain.cpu().numpy())
print(f'Nodes to explain: {len(nodes_to_explain)}  ({(data.y[nodes_to_explain]==1).sum().item()} fraud, {(data.y[nodes_to_explain]==0).sum().item()} legit)')


## GNNExplainer (T-GCN and GraphSAGE)

In [ ]:
print('--- T-GCN ---')
gnnexp_tgcn = run_gnnexplainer(tgcn, data, nodes_to_explain, 'T-GCN')
np.save(os.path.join(RESULTS_PATH, 'gnn_attrs_gnnexp_tgcn.npy'), gnnexp_tgcn)
print('--- SAGE ---')
gnnexp_sage = run_gnnexplainer(sage, data, nodes_to_explain, 'SAGE')
np.save(os.path.join(RESULTS_PATH, 'gnn_attrs_gnnexp_sage.npy'), gnnexp_sage)


## Integrated Gradients (T-GCN and GraphSAGE)

In [ ]:
print('--- T-GCN ---')
ig_tgcn = run_ig(tgcn, data, nodes_to_explain, 'T-GCN')
np.save(os.path.join(RESULTS_PATH, 'gnn_attrs_ig_tgcn.npy'), ig_tgcn)
print('--- SAGE ---')
ig_sage = run_ig(sage, data, nodes_to_explain, 'SAGE')
np.save(os.path.join(RESULTS_PATH, 'gnn_attrs_ig_sage.npy'), ig_sage)


## GraphLIME (T-GCN and GraphSAGE)

In [ ]:
print('--- T-GCN ---')
glime_tgcn = run_graphlime(tgcn, data, nodes_to_explain, 'T-GCN')
np.save(os.path.join(RESULTS_PATH, 'gnn_attrs_glime_tgcn.npy'), glime_tgcn)
print('--- SAGE ---')
glime_sage = run_graphlime(sage, data, nodes_to_explain, 'SAGE')
np.save(os.path.join(RESULTS_PATH, 'gnn_attrs_glime_sage.npy'), glime_sage)


## Top features and Jaccard consistency

In [ ]:
def get_top_features(imp_matrix, feature_names, n=10):
    mean_imp = np.abs(imp_matrix).mean(axis=0)
    idx = np.argsort(mean_imp)[::-1][:n]
    return [(feature_names[i], mean_imp[i]) for i in idx]

def jaccard(l1, l2, k=10):
    s1 = {f for f, _ in l1[:k]}
    s2 = {f for f, _ in l2[:k]}
    return len(s1 & s2) / len(s1 | s2) if (s1 | s2) else 0

configs = [
    ('GNNExp', 'T-GCN', gnnexp_tgcn), ('GNNExp', 'SAGE', gnnexp_sage),
    ('IG', 'T-GCN', ig_tgcn), ('IG', 'SAGE', ig_sage),
    ('GraphLIME', 'T-GCN', glime_tgcn), ('GraphLIME', 'SAGE', glime_sage),
]
results = {f'{m}_{n}': get_top_features(a, feature_cols) for m, n, a in configs}

summary = []
for method, model_name, attrs in configs:
    for rank, (feat, imp) in enumerate(get_top_features(attrs, feature_cols), 1):
        summary.append({'Method': method, 'Model': model_name, 'Rank': rank, 'Feature': feat, 'Importance': imp})
pd.DataFrame(summary).to_csv(os.path.join(RESULTS_PATH, 'gnn_xai_feature_importance.csv'), index=False)

print('--- Cross-model Jaccard (same method) ---')
for method in ['GNNExp', 'IG', 'GraphLIME']:
    print(f'  {method}: T-GCN vs SAGE = {jaccard(results[method + "_T-GCN"], results[method + "_SAGE"]):.3f}')
print('--- Cross-method Jaccard (same model) ---')
for model_name in ['T-GCN', 'SAGE']:
    for m1, m2 in [('GNNExp', 'IG'), ('GNNExp', 'GraphLIME'), ('IG', 'GraphLIME')]:
        print(f'  {model_name}: {m1} vs {m2} = {jaccard(results[m1 + "_" + model_name], results[m2 + "_" + model_name]):.3f}')
